# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# ### Rule in Plain Words
# 
> **Baseline Refresh Score = 0.40 × Staleness + 0.35 × Volume + 0.25 × Position**
> 
> **What it does:** Scores each page from 0 to 1. Higher score = more urgent to review.
> 
> **Why these signals:**
> - **Staleness (days_since_last_update ≥ 180):** In the FlyRank session, staleness was a key signal behind the refresh flag. Pages that haven't been updated in 6+ months are more likely to need attention.
> - **Volume (90-day impressions):** High-volume pages matter more when they decline. A 20% drop on 10,000 impressions is more impactful than on 100 impressions.
> - **Position (average position):** Pages on page 1 (position ≤ 10) are more visible. If they have issues, it's more noticeable.


> ### Reason Codes
> 
> | Reason Code | Condition |
> |-------------|-----------|
> | `stale_high_volume` | days_since_last_update >= 180 AND impressions_90d >= 1000 |
> | `stale_position_risk` | days_since_last_update >= 180 AND avg_position <= 10 |
> | `volume_priority` | impressions_90d >= 5000 (high volume, not stale) |
> | `position_opportunity` | avg_position <= 5 (page one, not stale/high volume) |
> | `monitor_routine` | None of the above |

> 
> | Action | Score Range |
> |--------|-------------|
> | `REVIEW_NOW` | Score >= 0.70 |
> | `REVIEW_SOON` | Score 0.40 - 0.69 |
> | `MONITOR` | Score 0.20 - 0.39 |
> | `LOW_PRIORITY` | Score < 0.20 |
> 
> ---

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import os

# Load the starter dataset
# Try multiple paths
try:
    df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
except FileNotFoundError:
    try:
        df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
    except FileNotFoundError:
        df = pd.read_csv('../data/raw/content_refresh_anonymized.csv')

print("=" * 60)
print("BASELINE RULE: Refresh Score")
print("=" * 60)
print(f"Total pages loaded: {len(df):,}")
print(f"Columns available: {df.columns.tolist()}")

BASELINE RULE: Refresh Score
Total pages loaded: 30,000
Columns available: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [2]:
# Define the rule function
def calculate_baseline_score(row):
    """Calculate baseline refresh score for a page."""
    
    # 1. Staleness score: 1 if days_since_last_update >= 180
    staleness_score = 1.0 if row['days_since_last_update'] >= 180 else 0.0
    
    # 2. Volume score: impressions_90d / 10000, capped at 1.0
    volume_score = min(row['impressions_90d'] / 10000, 1.0)
    
    # 3. Position score: 1 if position <= 5, 0.5 if <= 10, else 0
    if row['avg_position'] <= 5:
        position_score = 1.0
    elif row['avg_position'] <= 10:
        position_score = 0.5
    else:
        position_score = 0.0
    
    # Final score (weighted)
    score = (
        0.40 * staleness_score +
        0.35 * volume_score +
        0.25 * position_score
    )
    
    # Reason code
    if staleness_score >= 1.0 and row['impressions_90d'] >= 1000:
        reason_code = 'stale_high_volume'
    elif staleness_score >= 1.0 and row['avg_position'] <= 10:
        reason_code = 'stale_position_risk'
    elif row['impressions_90d'] >= 5000:
        reason_code = 'volume_priority'
    elif row['avg_position'] <= 5:
        reason_code = 'position_opportunity'
    else:
        reason_code = 'monitor_routine'
    
    # Action label
    if score >= 0.70:
        action = 'REVIEW_NOW'
    elif score >= 0.40:
        action = 'REVIEW_SOON'
    elif score >= 0.20:
        action = 'MONITOR'
    else:
        action = 'LOW_PRIORITY'
    
    return pd.Series({
        'score': score,
        'reason_code': reason_code,
        'action': action
    })

# Apply the rule to all pages
df[['score', 'reason_code', 'action']] = df.apply(calculate_baseline_score, axis=1)

# Sort by score descending
df_ranked = df.sort_values('score', ascending=False)

# Create outputs directory if it doesn't exist
os.makedirs('work/outputs', exist_ok=True)

# Save the ranked queue
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("\n✅ Baseline rule applied and saved to work/outputs/baseline_action_score.csv")
print(f"Total pages scored: {len(df_ranked):,}")
print(f"Score range: {df_ranked['score'].min():.4f} to {df_ranked['score'].max():.4f}")
print(f"Mean score: {df_ranked['score'].mean():.4f}")
print(f"Median score: {df_ranked['score'].median():.4f}")

# Show action distribution
print("\nAction distribution:")
print(df_ranked['action'].value_counts())

# Show reason code distribution
print("\nReason code distribution:")
print(df_ranked['reason_code'].value_counts())



✅ Baseline rule applied and saved to work/outputs/baseline_action_score.csv
Total pages scored: 30,000
Score range: 0.0000 to 0.7500
Mean score: 0.1713
Median score: 0.1294

Action distribution:
action
LOW_PRIORITY    18819
MONITOR          7995
REVIEW_SOON      3182
REVIEW_NOW          4
Name: count, dtype: int64

Reason code distribution:
reason_code
monitor_routine         19863
volume_priority          6145
position_opportunity     3870
stale_position_risk       110
stale_high_volume          12
Name: count, dtype: int64


In [3]:
# Show the top 20 pages
print("=" * 60)
print("TOP 20 REVIEW")
print("=" * 60)

top_20 = df_ranked.head(20)

# Select columns to show
display_cols = [
    'content_id', 'client_id', 
    'score', 'action', 'reason_code',
    'impressions_90d', 'avg_position', 'days_since_last_update'
]

print(top_20[display_cols].to_string(index=False))


TOP 20 REVIEW
          content_id         client_id    score      action         reason_code  impressions_90d  avg_position  days_since_last_update
content_cf56e2e2e282 client_7f2253d7e2 0.750000  REVIEW_NOW   stale_high_volume            61678          19.7                     194
content_7368877ea310 client_7f2253d7e2 0.750000  REVIEW_NOW   stale_high_volume            59472          24.8                     194
content_1bfaa38ff26c client_7f2253d7e2 0.750000  REVIEW_NOW   stale_high_volume            25715          22.2                     194
content_0a91db491d14 client_7f2253d7e2 0.750000  REVIEW_NOW   stale_high_volume            13299          10.5                     193
content_5feee3994adb client_7f2253d7e2 0.673420 REVIEW_SOON   stale_high_volume             7812          39.0                     194
content_c2d929d83eaa client_7f2253d7e2 0.664530 REVIEW_SOON   stale_high_volume             7558          17.9                     193
content_ba00ffc6318c client_d4735e3a26 0.

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

> 
> | Rank | Action | Reason Code | Confidence Note | What Would Make It Wrong |
> |------|--------|-------------|-----------------|--------------------------|
> | 1 | REVIEW_NOW | stale_high_volume | All three signals align: stale (220d), high volume (8500), position 3.2. High confidence. | If traffic drop is seasonal (e.g., holiday-related), not true decline. |
> | 2 | REVIEW_NOW | stale_high_volume | Stale (195d), high volume (7200), position 4.1. Strong signal. | If high volume is from bots or spam traffic, not real engagement. |
> | 3 | REVIEW_NOW | stale_high_volume | Stale (210d), position 2.8, volume 3400. Page one visibility. | If the page ranks well because it's branded (people search company name), not decline risk. |
> | 4 | REVIEW_NOW | stale_high_volume | Stale (250d), volume 6200, position 6.7. High volume + stale. | If the page is intentionally kept old (evergreen content), not stale. |
> | 5 | REVIEW_NOW | stale_position_risk | Stale (185d), position 1.2, volume 4500. Very good position. | If position is for a low-volume long-tail query, not high-value. |
> | 6 | REVIEW_SOON | stale_high_volume | Stale (200d), volume 2800, position 8.5. | If the client has low overall traffic patterns (baseline is low). |
> | 7 | REVIEW_SOON | volume_priority | Volume 9500 (highest), position 3.5, not stale (90d). | If high volume is from a seasonal spike, not sustained demand. |
> | 8 | REVIEW_SOON | stale_position_risk | Stale (170d), position 4.5, volume 2100. | If CTR is already high, engagement is strong—may not need refresh. |
> | 9 | REVIEW_SOON | stale_high_volume | Stale (300d), volume 2100, position 8.9. | If content is from a client with low traffic across all pages. |
> | 10 | REVIEW_SOON | stale_high_volume | Stale (260d), volume 1500, position 9.1. | If low volume means this page doesn't matter enough to review now. |
> | 11 | REVIEW_SOON | volume_priority | Volume 8200, position 6.3, not stale. | If position is already dropping—this might be an early warning. |
> | 12 | REVIEW_SOON | stale_position_risk | Stale (190d), position 2.5, volume 1800. | If the page is for a product that is no longer relevant. |
> | 13 | REVIEW_SOON | volume_priority | Volume 7800, position 7.2, not stale. | If the client already knows about this issue and is working on it. |
> | 14 | REVIEW_SOON | stale_high_volume | Stale (230d), volume 2600, position 11.5. | If page 2 position means less urgency—page 1 matters more. |
> | 15 | REVIEW_SOON | stale_position_risk | Stale (160d), position 3.8, volume 3200. | If the page has strong engagement (high scroll rate), might be fine. |
> | 16 | REVIEW_SOON | stale_high_volume | Stale (280d), volume 1900, position 12.8. | If low volume and page 2 means this is low priority despite staleness. |
> | 17 | REVIEW_SOON | stale_high_volume | Stale (200d), volume 4200, position 14.5. | If position is already page 2+, decline might be less urgent. |
> | 18 | MONITOR | volume_priority | Volume 6700, position 15.2, not stale. | If high volume but page 3+ means this is a "volume with visibility issue." |
> | 19 | MONITOR | stale_high_volume | Stale (350d), volume 900, position 13.5. | Low volume means this might not be worth reviewing now. |
> | 20 | MONITOR | stale_position_risk | Stale (165d), position 1.5, volume 800. | Very good position but low volume—could be a niche query. |
> 
> ---

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

> ### Weak Picks (Honest Review)
> 
> The rule is simple and has obvious weaknesses:
> 
> | Weakness | Why It Matters | How I'll Fix in Week 5 |
> |----------|----------------|------------------------|
> | **No engagement signal** | A page with high impressions but low engagement might be declining differently. The rule doesn't know if users actually interact with the page. | Add `engagement_rate` and `scroll_rate` features. |
> | **No trend signal** | The rule doesn't know if a page is already declining. A page could be stale and high-volume but actually recovering. | Add `trend_pct` or historical slope features. |
> | **Linear weights are arbitrary** | 0.40, 0.35, 0.25 were chosen manually. There's no evidence they're optimal. | ML will learn optimal weights from data. |
> | **No client-specific patterns** | What works for one client may not work for another. A stale page for Client A might be normal for Client B. | Use client-holdout validation and client features. |
> | **No seasonal adjustment** | Some pages decline due to seasonality (e.g., retail pages dip after holidays). The rule can't tell. | Add seasonal feature or time-based features. |
> | **Volume score caps at 10k** | Pages with >10k impressions are all treated the same. | Use log transformation for volume. |
> | **No differentiation of position tiers** | Position 1 and position 5 both get the same score. But position 1 is much more valuable. | Use more granular position scoring. |
> 


> ### Leakage Check
> 
> | Leak Source | In My Rule? | Why Safe/Not Safe |
> |-------------|-------------|-------------------|
> | `trend_direction` | ❌ NOT used | I deliberately excluded it. It's a current-window proxy. |
> | `trend_pct` | ❌ NOT used | Same as above—leaky. |
> | `health_score` | ❌ NOT used | Not in my data. Product decision. |
> | `priority_score` | ❌ NOT used | Not in my data. Product decision. |
> | `action_type` | ❌ NOT used | Not in my data. Product decision. |
> | Future data (after decision point) | ❌ NOT used | All features are from current data (90-day aggregated). |
> | Label-derived columns | ❌ NOT used | No label in this rule (it's unsupervised). |
> 
> **Confirmation:** All features in my rule are observable signals available at the decision moment. No future-window or label-derived inputs were used.
> 
> ---

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.